# Tables and figures for the SDI-C manuscript

Regenerates every table and figure in one file. No Google Drive: everything is written to
`/content/outputs` and downloaded as a single zip at the end.

## What this notebook does and does not do

**It does not re-run the experiments.** Fifteen fine-tuning runs and five backbones cannot be
reproduced in a notebook cell, so the *measured* quantities are entered as source data in Cell 2,
taken from the manuscript.

**It does recompute everything derivable from them.** Confidence intervals, standard-error
ratios, the crossed decomposition, the power curve and the design curve are all recomputed from
the measured inputs rather than transcribed, and Cell 4 checks each recomputed value against the
published one and prints a pass/fail table. That makes this an audit of the paper's arithmetic,
not a retyping of it.

Every cell is labelled **MEASURED** or **DERIVED** so it is unambiguous which numbers came from
an experiment and which came from a formula.

## Springer artwork compliance

The figures are regenerated to the journal's stated requirements rather than to their current
appearance. Three differences from the versions now in the manuscript:

1. **No titles inside the figures.** The guidelines say plainly: *do not include titles or
   captions within your illustrations*. Figures 2, 3 and 4 currently carry in-figure titles.
   They are removed here; the information belongs in the caption, which already carries it.
2. **Column widths.** 84 mm for single-panel figures, 174 mm for wide ones, height under 234 mm.
3. **Accessibility.** Hatching is added wherever colour carries meaning, so the figures survive
   greyscale printing and colour-blind readers.

Output formats: EPS (the preferred vector format), PDF, and PNG at 600 dpi, named `Fig1`,
`Fig2`, `Fig3`, `Fig4` as the guidelines require.

## 1. Setup

In [ ]:
import os, json, zipfile, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch

OUT = Path("/content/outputs")
(OUT / "tables").mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

# Springer: sans-serif lettering, 8-12 pt, fonts embedded in vector output.
MM = 1 / 25.4
W1, W2 = 84 * MM, 174 * MM          # single-column and double-column widths
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "grid.linewidth": 0.4, "lines.linewidth": 1.2,
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,   # embed fonts, do not outline
})

def save_figure(fig, n):
    """Write EPS, PDF and 600-dpi PNG under the Fig<N> names the guidelines require."""
    stem = OUT / "figures" / f"Fig{n}"
    for ext, kw in [("eps", {}), ("pdf", {}), ("png", {"dpi": 600})]:
        fig.savefig(f"{stem}.{ext}", **kw)
    w, h = fig.get_size_inches()
    print(f"  Fig{n}: {w/MM:.0f} x {h/MM:.0f} mm  ->  eps, pdf, png@600dpi"
          + ("" if h / MM <= 234 else "   [WARN] taller than the 234 mm limit"))
    plt.close(fig)

print("output directory:", OUT)

output directory: /content/outputs


## 2. Source data  — **MEASURED**

Everything in this cell comes from an experiment and cannot be recomputed here. Everything in
later cells is derived from it. Design parameters first: `F` is the number of held-out corruption
families and `m` the number of images per family.

In [ ]:
F = 6            # held-out corruption families
M_NEU = 900      # images per family, NEU-CLS
M_MT = 1344      # images per family, Magnetic Tile
ALPHA = 0.05

# --- Table 1: the corruption suite (design, not measurement) -----------------
TABLE1 = [
    ("Defocus blur", "Working-distance drift, autofocus failure", "train"),
    ("Motion blur", "Strip speed exceeding exposure time", "test"),
    ("Shot noise", "Photon-limited short exposure", "test"),
    ("Read noise", "Sensor amplifier noise at high gain", "train"),
    ("Brightness drift", "LED array output decay", "test"),
    ("Contrast loss", "Steam, oil mist, haze in the light path", "test"),
    ("Vignetting", "Optical falloff, lamp ageing at field edge", "test"),
    ("Banding", "Line-scan synchronisation error", "train"),
    ("JPEG artefacts", "Frame-grabber compression", "train"),
    ("Window contamination", "Dust or oil on the protective window", "train"),
    ("Vibration jitter", "Gantry mechanical resonance", "test"),
    ("Perspective drift", "Camera mount settling", "train"),
]

# --- Table 3: calibration arms, NEU-CLS, ResNet-50 ---------------------------
TABLE3 = [
    ("C0 uncalibrated",            "1.126 ± 0.093", "0.123 ± 0.010", 0.091, "– / –"),
    ("C1 scalar, clean fit",       "1.079 ± 0.180", "0.112 ± 0.019", 0.088, "no / no"),
    ("C2 scalar, augmented fit",   "0.779 ± 0.025", "0.038 ± 0.012", 0.095, "– / –"),
    ("C3 MLP, raw q",              "2.408 ± 0.566", "0.126 ± 0.013", 0.112, "no / no"),
    ("C4 linear, raw q",           "0.789 ± 0.180", "0.033 ± 0.007", 0.069, "no / no"),
    ("C5 linear, class one-hot",   "0.863 ± 0.136", "0.043 ± 0.009", 0.087, "no / no"),
    ("C6c linear, standardised q", "1.030 ± 0.297", "0.047 ± 0.011", 0.090, "no / no"),
    ("C7 bounded, standardised q", "0.869 ± 0.135", "0.041 ± 0.008", 0.075, "no / no"),
    ("C7q bounded, raw q",         "0.763 ± 0.084", "0.033 ± 0.005", 0.068, "yes / no"),
    ("C8 linear, class + q",       "0.945 ± 0.258", "0.043 ± 0.016", 0.075, "no / no"),
    ("C8b bounded, class + q",     "0.821 ± 0.134", "0.038 ± 0.013", 0.069, "no / no"),
]

# --- Table 4: per-family NLL differences (drives Figure 2) -------------------
FAMILIES6 = ["Motion\nblur", "Shot\nnoise", "Bright.\ndrift",
             "Contrast\nloss", "Vignet-\nting", "Vibr.\njitter"]
TABLE4 = {
    "C4 vs. C5":  [-0.458, -0.122, -0.073, -0.074, +0.434, -0.155],
    "C7q vs. C2": [-0.057, -0.103, -0.093, -0.121, +0.272, +0.004],
    "C7q vs. C5": [-0.457, -0.112, -0.075, -0.077, +0.280, -0.159],
    "C8b vs. C5": [-0.090, -0.137, -0.065, -0.070, +0.197, -0.084],
}
REVERSING_FAMILY = 4          # index of vignetting

# --- Table 5: variance components, twelve comparisons (sigma_a, sigma_e, rho)
TABLE5 = [
    ("C4 vs. C2",  0.313, 1.815, 0.029), ("C8b vs. C2", 0.178, 0.981, 0.032),
    ("C4 vs. C5",  0.211, 0.947, 0.047), ("C7q vs. C5", 0.233, 0.984, 0.053),
    ("C6c vs. C2", 0.208, 0.721, 0.077), ("C6c vs. C5", 0.286, 0.980, 0.079),
    ("C8b vs. C5", 0.119, 0.406, 0.079), ("C7q vs. C2", 0.148, 0.428, 0.106),
    ("C8 vs. C5",  0.503, 1.103, 0.172), ("C7 vs. C2",  0.811, 1.405, 0.250),
    ("C8 vs. C2",  0.430, 0.738, 0.253), ("C7 vs. C5",  0.872, 1.315, 0.305),
]

# --- Table 6: crossed decomposition; "observed" is the bootstrap ratio -------
TABLE6 = [("C4 vs. C2", 0.044, 0.040, 0.126, 8.23),
          ("C4 vs. C5", 0.214, 0.304, 0.845, 5.89)]

# --- Table 7: five backbones against two baselines --------------------------
TABLE7 = [("ConvNeXt-T", "C2", 0.047), ("ResNet-50", "C2", 0.069),
          ("DINOv2 ViT-S/14", "C2", 0.087), ("ViT-B/16", "C2", 0.092),
          ("EfficientNet-B0", "C2", 0.101), ("ConvNeXt-T", "C5", 0.033),
          ("ViT-B/16", "C5", 0.033), ("ResNet-50", "C5", 0.036),
          ("EfficientNet-B0", "C5", 0.062), ("DINOv2 ViT-S/14", "C5", 0.072)]

# Magnetic Tile enters only as summary statistics: the per-comparison values
# are not reported in the manuscript, so its row cannot be derived here.
MT_SUMMARY = {"rho_min": 0.018, "rho_max": 0.325, "rho_median": 0.118}

# --- Table 10: fine-tuning control ------------------------------------------
TABLE10 = [("Frozen probe", "paired NLL (C4 vs. C2)", 0.069, M_NEU),
           ("Frozen probe", "accuracy deficit",       0.173, M_NEU),
           ("Fine-tuned",   "accuracy deficit",       0.313, M_NEU)]

# --- Table 11: suite comparison ---------------------------------------------
TABLE11 = [("DINOv2 ViT-S/14", 0.849, 0.761), ("ViT-B/16", 0.833, 0.709),
           ("ConvNeXt-T", 0.818, 0.690), ("EfficientNet-B0", 0.761, 0.675),
           ("ResNet-50", 0.738, 0.604)]

# --- Figure 3 and 4 parameters ----------------------------------------------
EFFECT_SIZE_D = 0.425        # observed across families
RHO_MEDIAN_T5 = 0.079        # median rho of Table 5, anchors Figure 4
RHO_LO_T5, RHO_HI_T5 = 0.029, 0.305

print("source data loaded:",
      f"{len(TABLE5)} variance-component rows, {len(TABLE7)} backbone rows,"
      f" {len(TABLE4)} per-family comparisons")

source data loaded: 12 variance-component rows, 10 backbone rows, 4 per-family comparisons


## 3. Estimators  — **DERIVED**

In [ ]:
from scipy.stats import f as fdist, nct, t as tdist

def icc_ci(rho, m, F=F, alpha=ALPHA):
    """Exact one-way random-effects interval on the intraclass correlation."""
    Fstat = 1 + m * rho / (1 - rho)
    d1, d2 = F - 1, F * (m - 1)
    lo = (Fstat / fdist.ppf(1 - alpha / 2, d1, d2) - 1) / \
         (Fstat / fdist.ppf(1 - alpha / 2, d1, d2) + m - 1)
    hi = (Fstat / fdist.ppf(alpha / 2, d1, d2) - 1) / \
         (Fstat / fdist.ppf(alpha / 2, d1, d2) + m - 1)
    return lo, hi

def se_ratio(rho, m):
    """Equation (2) under the two-component reduction (sigma_b^2 = 0)."""
    return float(np.sqrt(1 + m * rho / (1 - rho)))

def se_ratio_crossed(sa, sb, se, m, F=F):
    """Equation (2) in full: sqrt((m*sa^2 + se^2) / (F*sb^2 + se^2))."""
    return float(np.sqrt((m * sa**2 + se**2) / (F * sb**2 + se**2)))

def power_1samp(d, n, alpha=ALPHA):
    """Power of a one-sample t test at effect size d with n families."""
    df, ncp = n - 1, d * np.sqrt(n)
    crit = tdist.ppf(1 - alpha / 2, df)
    return float(1 - nct.cdf(crit, df, ncp) + nct.cdf(-crit, df, ncp))

print("estimators defined")

estimators defined


## 4. Verification  — **DERIVED vs published**

Each recomputed value is compared with the number printed in the manuscript. Anything that does
not agree is flagged rather than silently overwritten.

In [ ]:
# The manuscript prints rho to three decimals and standard-error ratios to one.
# A recomputation from the PRINTED rho therefore cannot be expected to match the
# printed interval exactly: the true rho is only known to lie within its display
# precision. The check below asks the right question, which is whether the
# published value is attainable from SOME rho consistent with what is printed,
# rather than comparing against an arbitrary tolerance.
checks, failures = [], 0

def record(label, ok, shown, published):
    global failures
    if not ok:
        failures += 1
    checks.append((label, shown, published, "ok" if ok else "NOT ATTAINABLE"))

def attainable(fn, rho_disp, m, dp, published, half=0.0005, n=201):
    vals = [fn(r, m) for r in np.linspace(rho_disp - half, rho_disp + half, n)]
    ok = any(round(v, dp) == round(published, dp) for v in vals)
    return ok, f"[{min(vals):.4f}, {max(vals):.4f}]"

def check_tol(label, computed, published, tol):
    record(label, abs(computed - published) <= tol, f"{computed:.4f}", published)

# --- Tables 5, 7, 10: intervals and ratios derived from a printed rho --------
PUB_T5 = [((0.011, 0.156), 5.28), ((0.012, 0.169), 5.55), ((0.018, 0.233), 6.74),
          ((0.021, 0.255), 7.17), ((0.031, 0.336), 8.72), ((0.031, 0.341), 8.84),
          ((0.032, 0.343), 8.84), ((0.044, 0.420), 10.38), ((0.074, 0.557), 13.71),
          ((0.114, 0.668), 17.35), ((0.116, 0.672), 17.49), ((0.146, 0.726), 19.90)]
for (name, sa, se_, rho), ((plo, phi), pse) in zip(TABLE5, PUB_T5):
    ok, rng = attainable(lambda r, m: icc_ci(r, m)[0], rho, M_NEU, 3, plo)
    record(f"T5 {name} CI lower", ok, rng, plo)
    ok, rng = attainable(lambda r, m: icc_ci(r, m)[1], rho, M_NEU, 3, phi)
    record(f"T5 {name} CI upper", ok, rng, phi)
    ok, rng = attainable(se_ratio, rho, M_NEU, 2, pse)
    record(f"T5 {name} SE ratio", ok, rng, pse)

PUB_T7 = [(0.018, 0.231, 6.7), (0.028, 0.312, 8.2), (0.035, 0.366, 9.3), (0.037, 0.380, 9.6),
          (0.042, 0.406, 10.1), (0.012, 0.173, 5.6), (0.013, 0.175, 5.6), (0.014, 0.185, 5.8),
          (0.025, 0.288, 7.8), (0.029, 0.320, 8.4)]
for (bb, base, rho), (plo, phi, pse) in zip(TABLE7, PUB_T7):
    ok, rng = attainable(lambda r, m: icc_ci(r, m)[0], rho, M_NEU, 3, plo)
    record(f"T7 {bb} vs {base} CI lower", ok, rng, plo)
    ok, rng = attainable(se_ratio, rho, M_NEU, 1, pse)
    record(f"T7 {bb} vs {base} SE ratio", ok, rng, pse)

for (reg, met, rho, m), (plo, phi, pse) in zip(
        TABLE10, [(0.028, 0.312, 8.2), (0.075, 0.558, 13.7), (0.150, 0.733, 20.3)]):
    ok, rng = attainable(lambda r, mm: icc_ci(r, mm)[0], rho, m, 3, plo)
    record(f"T10 {reg}/{met} CI lower", ok, rng, plo)
    ok, rng = attainable(lambda r, mm: icc_ci(r, mm)[1], rho, m, 3, phi)
    record(f"T10 {reg}/{met} CI upper", ok, rng, phi)
    ok, rng = attainable(se_ratio, rho, m, 1, pse)
    record(f"T10 {reg}/{met} SE ratio", ok, rng, pse)

# --- Table 8 extremes, which also depend on Magnetic Tile's m ----------------
ok, rng = attainable(se_ratio, MT_SUMMARY["rho_min"], M_MT, 1, 5.0)
record("T8 Magnetic Tile SE ratio min", ok, rng, 5.0)
ok, rng = attainable(se_ratio, MT_SUMMARY["rho_max"], M_MT, 1, 25.4)
record("T8 Magnetic Tile SE ratio max", ok, rng, 25.4)

# --- Table 6: derived from three printed sigmas, so a plain tolerance is right
for (name, sa, sb, se_, observed), pub in zip(TABLE6, [(0.101, 8.37), (0.054, 5.74)]):
    tot = sa**2 + sb**2 + se_**2
    check_tol(f"T6 {name} rho_family", sa**2 / tot, pub[0], 0.003)
    check_tol(f"T6 {name} ratio Eq(2)", se_ratio_crossed(sa, sb, se_, M_NEU), pub[1], 0.10)

# --- Figures 3 and 4 ---------------------------------------------------------
for n_, pub in [(6, 0.14), (12, 0.27), (20, 0.44), (30, 0.61)]:
    vals = [power_1samp(d_, n_) for d_ in np.linspace(EFFECT_SIZE_D - 0.0005,
                                                      EFFECT_SIZE_D + 0.0005, 51)]
    record(f"Fig3 power at n={n_}", any(round(v, 2) == pub for v in vals),
           f"[{min(vals):.4f}, {max(vals):.4f}]", pub)
n80 = next(n_ for n_ in range(2, 200) if power_1samp(EFFECT_SIZE_D, n_) >= 0.80)
record("Fig3 families for power 0.80", n80 == 46, str(n80), 46)
for m_, pub, dp in [(50, 2.3, 1), (900, 8.8, 1), (50000, 65, 0)]:
    ok, rng = attainable(se_ratio, RHO_MEDIAN_T5, m_, dp, pub)
    record(f"Fig4 ratio at m={m_}", ok, rng, pub)

rep = pd.DataFrame(checks, columns=["quantity", "recomputed (range over rho display)",
                                    "published", "verdict"])
pd.set_option("display.max_rows", 200, "display.width", 140)
print(rep.to_string(index=False))
print(f"\n{len(checks) - failures}/{len(checks)} published values are reproducible "
      f"from the stated estimator")
if failures:
    print("NOT ATTAINABLE above: the published value cannot be produced by any rho")
    print("consistent with its printed precision. Investigate before submitting.")
rep.to_csv(OUT / "tables" / "verification.csv", index=False)

                                        quantity recomputed (range over rho display)  published verdict
                           T5 C4 vs. C2 CI lower                    [0.0106, 0.0110]      0.011      ok
                           T5 C4 vs. C2 CI upper                    [0.1540, 0.1586]      0.156      ok
                           T5 C4 vs. C2 SE ratio                    [5.2347, 5.3251]      5.280      ok
                          T5 C8b vs. C2 CI lower                    [0.0118, 0.0122]      0.012      ok
                          T5 C8b vs. C2 CI upper                    [0.1675, 0.1720]      0.169      ok
                          T5 C8b vs. C2 SE ratio                    [5.5020, 5.5886]      5.550      ok
                           T5 C4 vs. C5 CI lower                    [0.0180, 0.0184]      0.018      ok
                           T5 C4 vs. C5 CI upper                    [0.2302, 0.2341]      0.233      ok
                           T5 C4 vs. C5 SE ratio                

## 5. Tables  — **DERIVED where derivable**

In [ ]:
def fmt_ci(rho, m):
    lo, hi = icc_ci(rho, m)
    return f"[{lo:.3f}, {hi:.3f}]"

T = {}

T[1] = pd.DataFrame(TABLE1, columns=["Family", "Physical cause", "Split"])

T[3] = pd.DataFrame(TABLE3, columns=["Arm", "NLL", "ECE", "AURC", "Beats C2? (image / family)"])

T[4] = pd.DataFrame(
    [[k] + [f"{v:+.3f}" for v in vals] for k, vals in TABLE4.items()],
    columns=["Comparison"] + [f.replace("\n", " ") for f in FAMILIES6])

T[5] = pd.DataFrame(
    [(n, f"{sa:.3f}", f"{se:.3f}", f"{r:.3f}", fmt_ci(r, M_NEU), f"{se_ratio(r, M_NEU):.2f}")
     for n, sa, se, r in TABLE5],
    columns=["Comparison", "σ_a", "σ_e", "ρ", "95% CI on ρ", "SE ratio (Eq. 2)"])

T[6] = pd.DataFrame(
    [(n, f"{sa:.3f}", f"{sb:.3f}", f"{se:.3f}",
      f"{sa**2/(sa**2+sb**2+se**2):.3f}",
      f"{se_ratio_crossed(sa, sb, se, M_NEU):.2f}", f"{obs:.2f}")
     for n, sa, sb, se, obs in TABLE6],
    columns=["Comparison", "σ_a", "σ_b", "σ_e", "ρ_family", "ratio (Eq. 2)", "observed"])

T[7] = pd.DataFrame(
    [(bb, base, f"{r:.3f}", fmt_ci(r, M_NEU), f"{se_ratio(r, M_NEU):.1f}")
     for bb, base, r in TABLE7],
    columns=["Backbone", "vs.", "ρ", "95% CI", "SE ratio"])

neu_rhos = [r for _, _, r in TABLE7]
T[8] = pd.DataFrame([
    ["NEU-CLS (balanced, textural)",
     f"{min(neu_rhos):.3f}–{max(neu_rhos):.3f}", f"{np.median(neu_rhos):.3f}",
     f"{se_ratio(min(neu_rhos), M_NEU):.1f}–{se_ratio(max(neu_rhos), M_NEU):.1f}×"],
    ["Magnetic Tile (imbalanced, localized)",
     f"{MT_SUMMARY['rho_min']:.3f}–{MT_SUMMARY['rho_max']:.3f}",
     f"{MT_SUMMARY['rho_median']:.3f}",
     f"{se_ratio(MT_SUMMARY['rho_min'], M_MT):.1f}–{se_ratio(MT_SUMMARY['rho_max'], M_MT):.1f}×"],
    ["Pooled",
     f"{MT_SUMMARY['rho_min']:.3f}–{MT_SUMMARY['rho_max']:.3f}", "0.087",
     f"{se_ratio(MT_SUMMARY['rho_min'], M_MT):.1f}–{se_ratio(MT_SUMMARY['rho_max'], M_MT):.1f}×"],
], columns=["Dataset", "ρ range", "median", "SE ratio"])

T[9] = pd.DataFrame([
    ["Comparisons", "10", "20"],
    ["Family heterogeneity ρ",
     f"{min(neu_rhos):.3f}–{max(neu_rhos):.3f}",
     f"{MT_SUMMARY['rho_min']:.3f}–{MT_SUMMARY['rho_max']:.3f}"],
    ["Median ρ", f"{np.median(neu_rhos):.3f}", "0.087"],
    ["SE inflation",
     f"{se_ratio(min(neu_rhos), M_NEU):.1f}–{se_ratio(max(neu_rhos), M_NEU):.1f}×",
     f"{se_ratio(MT_SUMMARY['rho_min'], M_MT):.1f}–{se_ratio(MT_SUMMARY['rho_max'], M_MT):.1f}×"],
    ["Intervals excluding zero", "10/10", "20/20"],
    ["Image-significant comparisons that reverse", "all", "all"],
], columns=["", "NEU-CLS only (primary)", "Both datasets (extension)"])

T[10] = pd.DataFrame(
    [(reg, met, f"{r:.3f}", fmt_ci(r, m), f"{se_ratio(r, m):.1f}×")
     for reg, met, r, m in TABLE10],
    columns=["Regime", "Metric", "ρ", "95% CI", "SE ratio"])

T[11] = pd.DataFrame(TABLE11, columns=["Backbone", "SDI-C", "ImageNet-C"])

for n, df in sorted(T.items()):
    df.to_csv(OUT / "tables" / f"Table{n:02d}.csv", index=False)
    (OUT / "tables" / f"Table{n:02d}.md").write_text(df.to_markdown(index=False))
    (OUT / "tables" / f"Table{n:02d}.tex").write_text(
        df.to_latex(index=False, escape=False, column_format="l" * df.shape[1]))
    print(f"Table {n}: {df.shape[0]} x {df.shape[1]}")

print("\nTable 2 (the thirteen audit checks) is prose, not data; it is left in the manuscript.")
print("\n--- Table 5 ---"); print(T[5].to_string(index=False))
print("\n--- Table 10 ---"); print(T[10].to_string(index=False))

Table 1: 12 x 3
Table 3: 11 x 5
Table 4: 4 x 7
Table 5: 12 x 6
Table 6: 2 x 7
Table 7: 10 x 5
Table 8: 3 x 4
Table 9: 6 x 3
Table 10: 3 x 5
Table 11: 5 x 3

Table 2 (the thirteen audit checks) is prose, not data; it is left in the manuscript.

--- Table 5 ---
Comparison   σ_a   σ_e     ρ    95% CI on ρ SE ratio (Eq. 2)
 C4 vs. C2 0.313 1.815 0.029 [0.011, 0.156]             5.28
C8b vs. C2 0.178 0.981 0.032 [0.012, 0.170]             5.55
 C4 vs. C5 0.211 0.947 0.047 [0.018, 0.232]             6.74
C7q vs. C5 0.233 0.984 0.053 [0.021, 0.255]             7.17
C6c vs. C2 0.208 0.721 0.077 [0.031, 0.337]             8.72
C6c vs. C5 0.286 0.980 0.079 [0.032, 0.343]             8.84
C8b vs. C5 0.119 0.406 0.079 [0.032, 0.343]             8.84
C7q vs. C2 0.148 0.428 0.106 [0.043, 0.418]            10.38
 C8 vs. C5 0.503 1.103 0.172 [0.074, 0.557]            13.71
 C7 vs. C2 0.811 1.405 0.250 [0.114, 0.668]            17.35
 C8 vs. C2 0.430 0.738 0.253 [0.116, 0.671]            17.49
 C7 vs. 

## 6. Figure 1  — the argument in one picture

In [ ]:
fig, ax = plt.subplots(figsize=(W2, 62 * MM))
ax.set_xlim(0, 10); ax.set_ylim(0, 6.2); ax.axis("off")

nfam, nimg = 6, 8
x0, y0, cw, ch = 1.15, 1.5, 0.42, 0.42

for i in range(nimg):
    for f in range(nfam):
        ax.add_patch(Rectangle((x0 + i * cw, y0 + f * ch), cw * 0.88, ch * 0.88,
                               facecolor="#dfe6ee", edgecolor="white", linewidth=0.6))

# an image-level resample: whole columns are redrawn, every family stays present
for i in (1, 4, 6):
    ax.add_patch(Rectangle((x0 + i * cw - 0.03, y0 - 0.06), cw * 0.94, nfam * ch + 0.02,
                           facecolor="none", edgecolor="#1f4e79", linewidth=1.3))
ax.annotate("", xy=(x0 + nimg * cw + 0.05, y0 - 0.30), xytext=(x0 - 0.05, y0 - 0.30),
            arrowprops=dict(arrowstyle="<->", color="#1f4e79", lw=1.0))
ax.text(x0 + nimg * cw / 2, y0 - 0.62, "image-level bootstrap\nresamples columns",
        ha="center", va="top", color="#1f4e79", fontsize=7.5)

ax.annotate("", xy=(x0 - 0.34, y0 + nfam * ch + 0.05), xytext=(x0 - 0.34, y0 - 0.05),
            arrowprops=dict(arrowstyle="<->", color="#a33", lw=1.0))
ax.text(x0 - 0.48, y0 + nfam * ch / 2, "corruption\nfamilies", rotation=90,
        ha="center", va="center", color="#a33", fontsize=7.5)

ax.text(x0 + nimg * cw / 2, y0 + nfam * ch + 0.28,
        "every image is evaluated under every family", ha="center", fontsize=8)

# consequence panel: one auto-sizing text box, so the frame always fits the text
msg = ("$\\bf{consequence}$\n\n"
       "The family effect is constant across a column,\n"
       "so it contributes no variance to an\n"
       "image-level resample.\n\n"
       "At $\\sigma_a^2 = 0$ the family-level standard error\n"
       "is at most the image-level one, so any observed\n"
       "widening cannot be an artefact of six clusters.\n\n"
       "The ratio grows as $\\sqrt{m}$: more images widen\n"
       "the gap, more families close it.")
ax.text(6.05, 4.95, msg, fontsize=6.8, va="top", ha="left", linespacing=1.6,
        bbox=dict(boxstyle="round,pad=0.55", facecolor="#f6f7f9",
                  edgecolor="#c8ced6", linewidth=0.7))

save_figure(fig, 1)

  Fig1: 174 x 62 mm  ->  eps, pdf, png@600dpi


## 7. Figure 2  — per-family differences

In [ ]:
fig, ax = plt.subplots(figsize=(W2, 66 * MM))
labels = list(TABLE4)
x = np.arange(len(FAMILIES6)); width = 0.2
# Colour encodes the comparison, so the legend means what it says. The reversing
# family is marked by a red edge and hatching, which also survives greyscale.
comp_colors = ["#c6dbef", "#6baed6", "#3182bd", "#08519c"]

for j, comp in enumerate(labels):
    for i, v in enumerate(TABLE4[comp]):
        reversing = (i == REVERSING_FAMILY)
        ax.bar(x[i] + (j - 1.5) * width, v, width * 0.92,
               color=comp_colors[j],
               hatch="///" if reversing else None,
               edgecolor="#b2182b" if reversing else "white",
               linewidth=0.9 if reversing else 0.4,
               label=comp if i == 0 else None)

ax.axhline(0, color="black", linewidth=0.7)
ax.set_xticks(x); ax.set_xticklabels(FAMILIES6)
ax.set_ylabel("$\\Delta$ NLL  (negative favours\nthe quality-conditioned arm)")
ax.grid(axis="y", alpha=0.25); ax.set_axisbelow(True)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
ax.legend(ncol=4, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.16))
ax.annotate("sign reverses in all four",
            xy=(x[REVERSING_FAMILY], 0.44), xytext=(x[REVERSING_FAMILY] - 1.6, 0.40),
            fontsize=7, color="#b2182b",
            arrowprops=dict(arrowstyle="->", color="#b2182b", lw=0.7))
save_figure(fig, 2)

  Fig2: 174 x 66 mm  ->  eps, pdf, png@600dpi


## 8. Figure 3  — power against number of families

In [ ]:
fig, ax = plt.subplots(figsize=(W1, 58 * MM))
ns = np.arange(3, 61)
pw = [power_1samp(EFFECT_SIZE_D, int(n)) for n in ns]

ax.plot(ns, pw, color="#1f4e79")
ax.axhline(0.80, color="#666", linestyle="--", linewidth=0.7)
ax.text(59, 0.815, "power 0.80", ha="right", fontsize=6.5, color="#444")

p6 = power_1samp(EFFECT_SIZE_D, F)
ax.plot([F], [p6], "o", color="#b2182b", markersize=4)
ax.annotate(f"SDI-C, {F} families\n(power {p6:.2f})", xy=(F, p6), xytext=(11, 0.16),
            fontsize=6.8, color="#b2182b",
            arrowprops=dict(arrowstyle="->", color="#b2182b", lw=0.7))
ax.plot([n80], [power_1samp(EFFECT_SIZE_D, n80)], "o", color="#1f4e79", markersize=3.5)
ax.annotate(f"{n80} families", xy=(n80, 0.80), xytext=(n80 - 16, 0.62),
            fontsize=6.8, color="#1f4e79",
            arrowprops=dict(arrowstyle="->", color="#1f4e79", lw=0.7))

ax.set_xlabel("number of test corruption families")
ax.set_ylabel("power")
ax.set_ylim(0, 1); ax.set_xlim(3, 60)
ax.grid(alpha=0.25); ax.set_axisbelow(True)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
save_figure(fig, 3)

  Fig3: 84 x 58 mm  ->  eps, pdf, png@600dpi


## 9. Figure 4  — overstatement against test-set size

In [ ]:
fig, ax = plt.subplots(figsize=(W1, 58 * MM))
ms = np.logspace(np.log10(20), np.log10(1e5), 300)

ax.fill_between(ms, [se_ratio(RHO_LO_T5, m) for m in ms],
                [se_ratio(RHO_HI_T5, m) for m in ms],
                color="#cfe0ef", alpha=0.85, linewidth=0,
                label=f"observed ICC range\n({RHO_LO_T5:.3f}–{RHO_HI_T5:.3f})")
ax.plot(ms, [se_ratio(RHO_MEDIAN_T5, m) for m in ms], color="#1f4e79",
        label=f"median ICC = {RHO_MEDIAN_T5:.3f}")
ax.axhline(1.0, color="#666", linestyle="--", linewidth=0.7)

ax.plot([M_NEU], [se_ratio(RHO_MEDIAN_T5, M_NEU)], "o", color="#b2182b", markersize=4)
ax.annotate(f"this study\n({M_NEU} images/family)",
            xy=(M_NEU, se_ratio(RHO_MEDIAN_T5, M_NEU)), xytext=(1500, 3.0),
            fontsize=6.8, color="#b2182b",
            arrowprops=dict(arrowstyle="->", color="#b2182b", lw=0.7))

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("images per corruption family")
ax.set_ylabel("SE$_{\\mathrm{family}}$ / SE$_{\\mathrm{image}}$")
ax.grid(alpha=0.25, which="both"); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper left", fontsize=6.5)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
save_figure(fig, 4)

  Fig4: 84 x 58 mm  ->  eps, pdf, png@600dpi


## 10. Collect and download

In [ ]:
manifest = {
    "design": {"F": F, "m_NEU_CLS": M_NEU, "m_Magnetic_Tile": M_MT, "alpha": ALPHA},
    "verification": {"checks": len(checks), "agreeing": len(checks) - failures,
                     "mismatches": failures},
    "figures": {"format": ["eps", "pdf", "png@600dpi"],
                "widths_mm": {"Fig1": 174, "Fig2": 174, "Fig3": 84, "Fig4": 84},
                "in_figure_titles": "removed, per the artwork guidelines"},
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))

zip_path = "/content/sdic_tables_and_figures.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            z.write(p, p.relative_to(OUT))
n = len(zipfile.ZipFile(zip_path).namelist())
print(f"{n} files, {os.path.getsize(zip_path)/1e6:.2f} MB -> {zip_path}")
if failures:
    print(f"\n[WARN] {failures} verification mismatches; see verification.csv")

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print(f"[note] automatic download unavailable ({type(e).__name__}); "
          f"use the file browser on the left")

44 files, 0.78 MB -> /content/sdic_tables_and_figures.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>